In [1]:
# IMPORTS

import json
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Define the path to the market JSONL file.
market_path = Path("../data/polymarket_markets_1y.jsonl")

# Define the path to the orderbook JSONL file.
orderbook_path = Path("../data/polymarket_orderbooks.jsonl")

# Print the paths so we can confirm them.
print(market_path)
print(orderbook_path)

..\data\polymarket_markets_1y.jsonl
..\data\polymarket_orderbooks.jsonl


In [2]:
# HELPERS

# Define a helper function to safely parse fields that are JSON-encoded strings.
def parse_json_list(value):
    # If the value is already a list, return it directly.
    if isinstance(value, list):
        return value
    
    # If the value is missing, return an empty list.
    if value is None:
        return []
    
    # If the value is a string, try to parse it as JSON.
    if isinstance(value, str):
        # Remove surrounding whitespace.
        value = value.strip()
        
        # Return an empty list for blank strings.
        if value == "":
            return []
        
        # Parse the string as JSON.
        return json.loads(value)
    
    # For anything unexpected, return an empty list.
    return []


# Define a helper function to convert fee strings into decimal form.
def parse_fee_decimal(value):
    # If the fee is missing, return NaN.
    if value is None:
        return np.nan
    
    # Try to convert the fee into a float scaled by 1e18.
    try:
        # Convert the string to integer first, then divide by 1e18.
        return int(value) / 1e18
    except Exception:
        # Return NaN if parsing fails.
        return np.nan


# Define a helper function to safely convert values to float.
def safe_float(value):
    # Return NaN if the value is missing.
    if value is None:
        return np.nan
    
    # Try to cast the value to float.
    try:
        return float(value)
    except Exception:
        return np.nan


# Define a helper function to compute best bid from a list of bid levels.
def get_best_bid(bids):
    # Return NaN if there are no bids.
    if not bids:
        return np.nan
    
    # Return the maximum bid price across all levels.
    return max(float(level["price"]) for level in bids)


# Define a helper function to compute best ask from a list of ask levels.
def get_best_ask(asks):
    # Return NaN if there are no asks.
    if not asks:
        return np.nan
    
    # Return the minimum ask price across all levels.
    return min(float(level["price"]) for level in asks)


# Define a helper function to get the size at the best bid.
def get_best_bid_size(bids):
    # Return NaN if there are no bids.
    if not bids:
        return np.nan
    
    # Find the bid level with the highest price.
    best_level = max(bids, key=lambda level: float(level["price"]))
    
    # Return its size as float.
    return float(best_level["size"])


# Define a helper function to get the size at the best ask.
def get_best_ask_size(asks):
    # Return NaN if there are no asks.
    if not asks:
        return np.nan
    
    # Find the ask level with the lowest price.
    best_level = min(asks, key=lambda level: float(level["price"]))
    
    # Return its size as float.
    return float(best_level["size"])


# Define a helper function to sum depth across the top n levels.
def get_depth_top_n(levels, n):
    # Return 0.0 if there are no levels.
    if not levels:
        return 0.0
    
    # Sort levels by price.
    sorted_levels = levels
    
    # Sum the size of the first n levels.
    return sum(float(level["size"]) for level in sorted_levels[:n])


# Define a helper function to compute imbalance.
def compute_imbalance(bid_depth, ask_depth):
    # Compute the denominator.
    denom = bid_depth + ask_depth
    
    # Return NaN if the denominator is zero.
    if denom == 0:
        return np.nan
    
    # Return the imbalance.
    return (bid_depth - ask_depth) / denom

In [3]:
# OUTPUT DIRECTORY DEFINITION

# Define the output directory for processed files.
output_dir = Path("../data/processed")

# Create the directory if it does not already exist.
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# TO CREATE MARKETS TABLE

# Create an empty list to collect market-level rows.
market_rows = []

# Open the market file for reading.
with open(market_path, "r", encoding="utf-8") as f:
    # Loop through each line in the file.
    for line in f:
        # Parse the current line as JSON.
        market = json.loads(line)
        
        # Parse the outcomes field, which is stored as a JSON string.
        outcomes = parse_json_list(market.get("outcomes"))
        
        # Parse the outcomePrices field, which is stored as a JSON string.
        outcome_prices = parse_json_list(market.get("outcomePrices"))
        
        # Build a clean market-level row.
        market_rows.append({
            "market_id": market.get("id"),
            "condition_id": market.get("conditionId"),
            "question": market.get("question"),
            "slug": market.get("slug"),
            "description": market.get("description"),
            "start_date": market.get("startDate"),
            "end_date": market.get("endDate"),
            "closed_time": market.get("closedTime"),
            "active": market.get("active"),
            "closed": market.get("closed"),
            "archived": market.get("archived"),
            "accepting_orders": market.get("acceptingOrders"),
            "enable_order_book": market.get("enableOrderBook"),
            "neg_risk": market.get("negRisk"),
            "fee_decimal": parse_fee_decimal(market.get("fee")),
            "volume": safe_float(market.get("volume")),
            "volume_clob": safe_float(market.get("volumeClob")),
            "market_best_bid": safe_float(market.get("bestBid")),
            "market_best_ask": safe_float(market.get("bestAsk")),
            "last_trade_price": safe_float(market.get("lastTradePrice")),
            "resolution_source": market.get("resolutionSource"),
            "uma_resolution_status": market.get("umaResolutionStatus"),
            "created_at": market.get("createdAt"),
            "updated_at": market.get("updatedAt"),
            "n_outcomes": len(outcomes),
            "outcomes_raw": json.dumps(outcomes),
            "outcome_prices_raw": json.dumps(outcome_prices),
        })

# Convert the list of dicts into a pandas DataFrame.
markets_df = pd.DataFrame(market_rows)

# Show the shape of the resulting table.
print(markets_df.shape)

# Show the first few rows.
markets_df.head()

In [ ]:
# TO SAVE MARKETS TABLE AS "markets.parquet"
markets_df.to_parquet(output_dir / "markets.parquet", index=False)

In [4]:
# TO READ MARKETS TABLE FROM "markets.parquet"
markets_df = pd.read_parquet(output_dir / "markets.parquet")
print(markets_df.shape)

(356736, 27)


In [28]:
btc_markets_df = markets_df[markets_df["question"].str.lower().str.contains("bitcoin")].dropna(subset=["question", "market_id", "end_date", "closed_time"])
btc_markets_filtered_df = btc_markets_df[btc_markets_df["end_date"] > "2026-01-01"].sort_values("volume", ascending=False).head(15)
relevant_market_ids = set(btc_markets_filtered_df["market_id"].astype(str))
len(relevant_market_ids)

15

In [ ]:
# TO CREATE MARKET-TOKENS TABLE

market_token_rows = []

with open(market_path, "r", encoding="utf-8") as f:
    for line in f:
        market = json.loads(line)

        outcomes = parse_json_list(market.get("outcomes"))
        outcome_prices = parse_json_list(market.get("outcomePrices"))
        token_ids = parse_json_list(market.get("clobTokenIds"))

        for i, token_id in enumerate(token_ids):
            market_token_rows.append({
                "market_id": market.get("id"),
                "token_id": str(token_id),
                "outcome_index": i,
                "outcome": outcomes[i] if i < len(outcomes) else None,
                "outcome_price_initial": safe_float(outcome_prices[i]) if i < len(outcome_prices) else np.nan,
                "question": market.get("question"),
                "slug": market.get("slug"),
                "description": market.get("description"),
                "start_date": market.get("startDate"),
                "end_date": market.get("endDate"),
                "closed_time": market.get("closedTime"),
                "active": market.get("active"),
                "closed": market.get("closed"),
                "archived": market.get("archived"),
                "accepting_orders": market.get("acceptingOrders"),
                "enable_order_book": market.get("enableOrderBook"),
                "neg_risk": market.get("negRisk"),
                "fee_decimal": parse_fee_decimal(market.get("fee")),
                "volume": safe_float(market.get("volume")),
                "volume_clob": safe_float(market.get("volumeClob")),
            })

market_tokens_df = pd.DataFrame(market_token_rows)
print(market_tokens_df.shape)
market_tokens_df.head()

In [ ]:
# TO SAVE MARKET-TOKENS TABLE AS "market_tokens.parquet"
market_tokens_df.to_parquet(output_dir / "market_tokens.parquet", index=False)

In [6]:
# TO READ MARKET-TOKENS TABLE
market_tokens_df = pd.read_parquet(output_dir / "market_tokens.parquet")
print(market_tokens_df.shape)

(713472, 20)


In [29]:
btc_tokens_filtered_df = market_tokens_df[
    market_tokens_df["market_id"].astype(str).isin(relevant_market_ids)
].copy()
btc_tokens_filtered_df.shape

(30, 20)

In [33]:
import os
import json
import asyncio
import aiohttp
import random
import time
from requests.exceptions import RequestException, HTTPError
from pathlib import Path

import requests
import pandas as pd
import numpy as np


API_KEY = "beb79777a5762ef81b41fbaae1dbb75d23fcee28"
DOME_HEADERS = {"Authorization": f"Bearer {API_KEY}", "x-api-key": API_KEY}

DOME_URL = "https://api.domeapi.io/v1/polymarket/orderbooks"
RETRYABLE_STATUS = {429, 500, 502, 503, 504}

async def extract_orderbook_from_df_async(df, window, interval, output_path, max_concurrent=5, max_retries=6):
    window = pd.Timedelta(window)
    interval = pd.Timedelta(interval)
    output_path = Path(output_path)

    def _to_utc_timestamp(value):
        if pd.isna(value) or value is None:
            return None
        ts = pd.Timestamp(value)
        return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

    def _choose_end_time(row):
        closed_time = row.get("closed_time", None)
        end_date = row.get("end_date", None)
        if pd.notna(closed_time):
            return _to_utc_timestamp(closed_time)
        if pd.notna(end_date):
            return _to_utc_timestamp(end_date)
        return None

    def _book_features(bids, asks):
        bids = bids or []
        asks = asks or []
        bids_sorted = sorted(bids, key=lambda x: float(x["price"]), reverse=True)
        asks_sorted = sorted(asks, key=lambda x: float(x["price"]))
        best_bid = float(bids_sorted[0]["price"]) if bids_sorted else np.nan
        best_ask = float(asks_sorted[0]["price"]) if asks_sorted else np.nan
        best_bid_size = float(bids_sorted[0]["size"]) if bids_sorted else np.nan
        best_ask_size = float(asks_sorted[0]["size"]) if asks_sorted else np.nan
        mid = (best_bid + best_ask) / 2 if pd.notna(best_bid) and pd.notna(best_ask) else np.nan
        spread = best_ask - best_bid if pd.notna(best_bid) and pd.notna(best_ask) else np.nan
        return {
            "best_bid": best_bid,
            "best_ask": best_ask,
            "mid": mid,
            "spread": spread,
            "best_bid_size": best_bid_size,
            "best_ask_size": best_ask_size,
            "n_bid_levels": len(bids_sorted),
            "n_ask_levels": len(asks_sorted),
        }

    async def _fetch_all_snapshots(session, token_id, start_ms, end_ms):
        params = {
            "token_id": str(token_id),
            "start_time": int(start_ms),
            "end_time": int(end_ms),
            "limit": 200,
        }
        all_snaps = []

        while True:
            for attempt in range(1, max_retries + 1):
                try:
                    async with session.get(DOME_URL, params=params, headers=DOME_HEADERS, timeout=30) as resp:
                        if resp.status in RETRYABLE_STATUS:
                            raise aiohttp.ClientResponseError(
                                request_info=resp.request_info,
                                history=resp.history,
                                status=resp.status,
                                message=f"retryable status={resp.status}",
                                headers=resp.headers,
                            )
                        resp.raise_for_status()
                        payload = await resp.json()
                        break
                except Exception as e:
                    if attempt == max_retries:
                        raise RuntimeError(f"Failed token_id={token_id} after {max_retries} retries: {e}")
                    backoff = min(30, (2 ** attempt) + random.random())
                    print(f"Retry {attempt}/{max_retries} token_id={token_id} in {backoff:.1f}s ({e})")
                    await asyncio.sleep(backoff)

            snaps = payload.get("snapshots", [])
            all_snaps.extend(snaps)

            pagination = payload.get("pagination", {}) or {}
            has_more = pagination.get("has_more", False)
            pagination_key = pagination.get("pagination_key") or pagination.get("paginationKey")
            if not has_more or not pagination_key:
                break

            params["pagination_key"] = pagination_key
            await asyncio.sleep(0.05)

        return all_snaps

    sem = asyncio.Semaphore(max_concurrent)
    records = df.to_dict("records")

    async def _process_row(i, row, session):
        token_id = str(row.get("token_id"))
        end_time = _choose_end_time(row)
        if end_time is None:
            return []

        start_time = end_time - window
        start_ms = int(start_time.timestamp() * 1000)
        end_ms = int(end_time.timestamp() * 1000)

        async with sem:
            print(f"Processing market_id={row.get('market_id')}, token_id={token_id}, {i}/{len(records)}")
            try:
                snapshots = await _fetch_all_snapshots(session, token_id, start_ms, end_ms)
            except Exception as e:
                print(f"Skipping token_id={token_id}: {e}")
                return []

        if not snapshots:
            return []

        snapshots = sorted(snapshots, key=lambda s: (s.get("timestamp", 0), s.get("indexedAt", 0)))
        out = []

        for snap in snapshots:
            snap_ts = pd.to_datetime(snap.get("timestamp"), unit="ms", utc=True, errors="coerce")

            bids = snap.get("bids", []) or []
            asks = snap.get("asks", []) or []
            feats = _book_features(bids, asks)

            out.append({
                "market_id": row.get("market_id"),
                "token_id": token_id,
                "outcome": row.get("outcome"),
                "question": row.get("question"),
                "slug": row.get("slug"),
                "window_start": start_time,
                "window_end": end_time,
                "snapshot_time": snap_ts,
                "snapshot_timestamp_ms": snap.get("timestamp"),
                "indexed_at_ms": snap.get("indexedAt"),
                "market_hash": snap.get("market"),
                "asset_id": snap.get("assetId"),
                "tick_size": float(snap["tickSize"]) if snap.get("tickSize") is not None else np.nan,
                "min_order_size": float(snap["minOrderSize"]) if snap.get("minOrderSize") is not None else np.nan,
                "orderbook_neg_risk": snap.get("negRisk"),
                **feats,
                "bids_json": json.dumps(bids),
                "asks_json": json.dumps(asks),
            })

        print(f"Processed {len(out)} rows for market_id={row.get('market_id')}, token_id={token_id}")
        return out

    connector = aiohttp.TCPConnector(limit=50, ttl_dns_cache=300)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [_process_row(i, row, session) for i, row in enumerate(records, start=1)]
        chunks = await asyncio.gather(*tasks)

    rows = [r for chunk in chunks for r in chunk]
    result_df = pd.DataFrame(rows)

    if result_df.empty:
        print("No rows extracted.")
        return result_df

    result_df = result_df.sort_values(
        ["market_id", "token_id", "snapshot_time"], kind="stable"
    ).reset_index(drop=True)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.suffix.lower() == ".parquet":
        result_df.to_parquet(output_path, index=False)
    elif output_path.suffix.lower() == ".csv":
        result_df.to_csv(output_path, index=False)
    else:
        output_path.mkdir(parents=True, exist_ok=True)
        for market_id, group in result_df.groupby("market_id"):
            group.to_parquet(output_path / f"market_{market_id}.parquet", index=False)

    print(f"Saved {len(result_df):,} rows")
    return result_df

In [34]:
features_output_path = output_dir / f"btc_15_markets_orderbook.parquet"

result_df = await extract_orderbook_from_df_async(
    df=btc_tokens_filtered_df,
    window="30D",
    interval="15m",
    output_path=features_output_path,
    max_concurrent=5,
)

Processing market_id=620015, token_id=4960072122879914031540087398045925546763584182114633893203762140348078315559, 1/30
Processing market_id=620015, token_id=78490642434213550855600439825729670852317345211936381977390919619271067452728, 2/30
Processing market_id=666217, token_id=20572254894388566434749427412990047074432984200146791571808682059731237563429, 3/30
Processing market_id=666217, token_id=10874559437709429248550318829337954716448627278767284238604313843895084433488, 4/30
Processing market_id=687200, token_id=92530351746682701626081429271272191534203523777966250895862560038889741880940, 5/30
Processed 6212 rows for market_id=666217, token_id=10874559437709429248550318829337954716448627278767284238604313843895084433488
Processing market_id=687200, token_id=15002165665234016876486282630507392757515779060349352387653810600083538604826, 6/30
Processed 6215 rows for market_id=666217, token_id=20572254894388566434749427412990047074432984200146791571808682059731237563429
Processing 

In [36]:
result_df["token_id"].nunique()

30

In [37]:
result_df = result_df.drop(
    columns=["market_id", "outcome", "question", "slug"],
    errors="ignore"
)

In [38]:
btc_joined_df = result_df.merge(
    btc_tokens_filtered_df,
    on = "token_id",
    how = "inner",
)

In [39]:
btc_joined_df.to_parquet(output_dir / "btc_15_markets_orderbook_joined.parquet", index=False)

In [41]:
btc_joined_df.columns

Index(['token_id', 'window_start', 'window_end', 'snapshot_time',
       'snapshot_timestamp_ms', 'indexed_at_ms', 'market_hash', 'asset_id',
       'tick_size', 'min_order_size', 'orderbook_neg_risk', 'best_bid',
       'best_ask', 'mid', 'spread', 'best_bid_size', 'best_ask_size',
       'n_bid_levels', 'n_ask_levels', 'bids_json', 'asks_json', 'market_id',
       'outcome_index', 'outcome', 'outcome_price_initial', 'question', 'slug',
       'description', 'start_date', 'end_date', 'closed_time', 'active',
       'closed', 'archived', 'accepting_orders', 'enable_order_book',
       'neg_risk', 'fee_decimal', 'volume', 'volume_clob'],
      dtype='object')

In [43]:
# Find 0/1 outcome pairs with arbitrage:
# BUY arb:  ask_0 + ask_1 < 1
# SELL arb: bid_0 + bid_1 > 1

ts_col = "snapshot_time"   # change if you prefer snapshot_timestamp_ms

q = btc_joined_df.copy()
q["best_bid"] = pd.to_numeric(q["best_bid"], errors="coerce")
q["best_ask"] = pd.to_numeric(q["best_ask"], errors="coerce")
q["outcome_index"] = pd.to_numeric(q["outcome_index"], errors="coerce")

# In case of duplicates per market/timestamp/outcome, keep best executable quotes:
# - for BUY arb use min ask
# - for SELL arb use max bid
agg = (
    q.groupby(["market_id", ts_col, "outcome_index"], as_index=False)
     .agg(best_bid=("best_bid", "max"),
          best_ask=("best_ask", "min"))
)

wide = agg.pivot(index=["market_id", ts_col], columns="outcome_index", values=["best_bid", "best_ask"])
wide.columns = [f"{a}_{int(b)}" for a, b in wide.columns]
wide = wide.reset_index()

# Keep only timestamps where both opposite outcomes (0 and 1) exist
needed = ["best_ask_0", "best_ask_1", "best_bid_0", "best_bid_1"]
opps = wide.dropna(subset=needed).copy()

opps["buy_sum"] = opps["best_ask_0"] + opps["best_ask_1"]
opps["sell_sum"] = opps["best_bid_0"] + opps["best_bid_1"]

opps["buy_arb"] = opps["buy_sum"] < 1.0
opps["sell_arb"] = opps["sell_sum"] > 1.0

arb_pairs = opps[opps["buy_arb"] | opps["sell_arb"]].copy()

print(f"Arb market/timestamp pairs: {len(arb_pairs):,}")
print(f"shape: {arb_pairs.shape}")
arb_pairs.head()

Arb market/timestamp pairs: 95
shape: (95, 10)


,market_id,snapshot_time,best_bid_0,best_bid_1,best_ask_0,best_ask_1,buy_sum,sell_sum,buy_arb,sell_arb
83014,1082781,2026-01-21 01:01:33.176000+00:00,0.18,0.920,0.080,0.82,0.900,1.100,True,True
83015,1082781,2026-01-21 01:01:34.085000+00:00,0.18,0.920,0.080,0.82,0.900,1.100,True,True
83016,1082781,2026-01-21 01:01:34.272000+00:00,0.17,0.920,0.080,0.83,0.910,1.090,True,True
83017,1082781,2026-01-21 01:01:34.328000+00:00,0.17,0.920,0.080,0.83,0.910,1.090,True,True
126762,1082796,2026-01-21 15:35:43.966000+00:00,0.10,0.901,0.099,0.90,0.999,1.001,True,True


In [44]:
btc_joined_df.shape

(488005, 40)

In [45]:
arb_pairs["snapshot_time"].unique()

<DatetimeArray>
['2026-01-21 01:01:33.176000+00:00', '2026-01-21 01:01:34.085000+00:00',
 '2026-01-21 01:01:34.272000+00:00', '2026-01-21 01:01:34.328000+00:00',
 '2026-01-21 15:35:43.966000+00:00', '2026-01-21 16:44:16.731000+00:00',
 '2026-01-21 19:32:00.314000+00:00', '2026-01-23 12:19:51.785000+00:00',
 '2026-01-31 16:38:56.595000+00:00', '2025-12-17 16:18:15.910000+00:00',
 '2025-12-27 01:15:55.005000+00:00', '2025-12-27 01:15:55.523000+00:00',
 '2025-12-27 01:15:56.125000+00:00', '2025-12-27 04:51:28.970000+00:00',
 '2025-12-27 04:58:25.319000+00:00', '2025-12-27 04:59:13.433000+00:00',
 '2025-12-27 05:02:43.043000+00:00', '2025-12-27 05:05:34.500000+00:00',
 '2025-12-27 05:07:43.458000+00:00', '2025-12-27 05:20:26.651000+00:00',
 '2025-12-27 05:21:18.889000+00:00', '2025-12-27 05:28:12.161000+00:00',
 '2025-12-27 05:36:08.316000+00:00', '2025-12-27 05:43:55.435000+00:00',
 '2025-12-27 05:47:11.875000+00:00', '2025-12-27 05:52:49.392000+00:00',
 '2025-12-27 05:53:11.766000+00:00'